# 01 — Exploratory data analysis

**Machine Learning · M.EIC001 · Group G51**

This notebook covers the **business understanding** recap and the **data understanding** phase of CRISP-DM. It ends with a list of decisions that feed the data preparation notebook.

## 0. Setup

In [23]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
DATA_DIR = Path("hockey_development_data") / "hockey_development_data"

## 1. Business understanding

**Business goals**

Before the start of each season clubs, fans and media all want to know the teams final standing are, and which teams are more likely to fire a coach. We’ll try to answer these questions with the 100 years of NHL dataset.
So our goals will be:
- Predict the final regular-season ranking of every team.
- Identify which teams are more likely to replace their coach during the season.
- Understand the reasons for what drives both outcomes.

We’ll be working with the following assumptions:
- Predictions are made the day before the first game, so nothing from the target season can be used as an input.
- The target is the conference rank, as agreed with the professor.
- No data from the outside can be used to enrich our prediction model.



**Data mining goals**

| Task | Target | DM task type |
|---|---|---|
| (a) | **Points percentage** of each team in the season (points won / points available), from which the final **conference** rank is derived by sorting within the conference | Regression, then ranking |
| (b) | Teams most likely to have a **mid-season** coach replacement | Binary classification |
| (c) | TBA | TBA |

We predict points percentage and not the rank directly, the rank is relative to the other teams in the conference, so it changes with the number of teams, while points percentage describes one team and is comparable across eras (raw points are not, because games per season and the points system changed). The predictions of all teams in a conference are sorted to obtain the rank, which is what we evaluate. Predicting the rank directly will still be a future possibility.

**Data mining success criteria**

| Task | Metrics | Baseline to beat |
|---|---|---|
| (a) | Spearman correlation between predicted and actual conference rank, and mean absolute rank error. MAE of points percentage for the model fit | Last season's rank (new teams get the mean rank) |
| (b) | Precision, recall, F1 and AUC (PR-AUC as well, because positives are rare) | "No coach change" and the base rate by era |

All models are evaluated on seasons later than the ones they were trained on.

## 2. Collect and inventory the data

The data comes as 22 csv files. In all of them `year` is the year the season started (2005-06 is stored as 2005), so the last season we have is 2009.

In [24]:
def load_tables(data_dir=DATA_DIR):
    return {p.stem: pd.read_csv(p, low_memory=False) for p in sorted(data_dir.glob("*.csv"))}

tables = load_tables()
print(len(tables), "tables loaded")

22 tables loaded


In [25]:
def inventory(tables):
    rows = []
    for name, df in tables.items():
        has_year = "year" in df.columns
        rows.append({
            "table": name,
            "rows": len(df),
            "cols": df.shape[1],
            "first_year": df["year"].min() if has_year else np.nan,
            "last_year": df["year"].max() if has_year else np.nan,
            "pct_missing": df.isna().mean().mean() * 100,
        })
    return pd.DataFrame(rows).set_index("table")

inventory(tables)

,rows,cols,first_year,last_year,pct_missing
table,,,,,
AwardsCoaches,75,5,1930.0,2009.0,20.000
AwardsMisc,120,6,1965.0,2009.0,19.722
AwardsPlayers,2021,6,1917.0,2009.0,22.464
Coaches,1741,14,1909.0,2009.0,20.998
CombinedShutouts,50,8,1929.0,2009.0,0.000
Goalies,4096,23,1909.0,2009.0,30.764
GoaliesSC,31,11,1912.0,1925.0,0.000
GoaliesShootout,346,8,2005.0,2009.0,0.000
HOF,356,4,1945.0,2009.0,0.000


Six tables go back to 1909: Teams, TeamSplits, TeamVsTeam, Coaches, Scoring and Goalies. The others only cover a few seasons (TeamsHalf stops in 1920 and the shootout tables start in 2005) and Master has no season because it describes people.

We only care about the NHL, so first we check which leagues are in the data.

In [26]:
league_rows = pd.DataFrame({
    t: tables[t].groupby("lgID").size() for t in ["Teams", "Coaches", "Scoring", "Goalies"]
})
league_span = tables["Teams"].groupby("lgID")["year"].agg(first_year="min", last_year="max")
league_rows.join(league_span).sort_values("first_year")

,Teams,Coaches,Scoring,Goalies,first_year,last_year
lgID,,,,,,
NHA,45,43,586,75,1909,1916
PCHA,41,41,404,47,1911,1923
NHL,1265,1522,40101,3678,1917,2009
WCHL,24,24,321,37,1921,1925
WHA,84,111,2436,259,1972,1978


In [27]:
nhl_years = tables["Teams"].query("lgID == 'NHL'")["year"].drop_duplicates().sort_values()
gaps = sorted(set(range(nhl_years.min(), nhl_years.max() + 1)) - set(nhl_years))

print(f"NHL seasons: {len(nhl_years)} ({nhl_years.min()}–{nhl_years.max()}), missing years in between: {gaps}")
print("NHL team-seasons:", int((tables["Teams"]["lgID"] == "NHL").sum()), "of", len(tables["Teams"]))
print("distinct tmID in the NHL:", tables["Teams"].query("lgID == 'NHL'")["tmID"].nunique(),
      "| distinct franchID:", tables["Teams"].query("lgID == 'NHL'")["franchID"].nunique())

NHL seasons: 92 (1917–2009), missing years in between: [2004]
NHL team-seasons: 1265 of 1459
distinct tmID in the NHL: 55 | distinct franchID: 36


What one row means in each group of tables:

| Group | Tables | One row is |
|---|---|---|
| Team-season | Teams, TeamsHalf, TeamSplits, TeamVsTeam | a team in a season (in half a season for TeamsHalf, against one opponent for TeamVsTeam) |
| Players and goalies | Master, Scoring, Goalies, ScoringShootout, GoaliesShootout | a person for Master, otherwise a player or goalie with a team in a season (`stint` counts the spells when a player is traded) |
| Coaches | Coaches | a coach with a team in a season |
| Playoffs and finals | SeriesPost, TeamsPost, TeamsSC, ScoringSC, GoaliesSC | a playoff series, a team's playoff totals in a season, or a team, skater or goalie in the Stanley Cup final (1912 to 1925 only) |
| Awards and misc | AwardsPlayers, AwardsCoaches, AwardsMisc, HOF, CombinedShutouts, ScoringSup, abbrev | an award given in a year, a Hall of Fame inductee, a shutout shared by two goalies, or a code and its full name for abbrev |

What we learned:
- The files have five leagues: NHA, PCHA, WCHL, WHA and the NHL. The NHL has 1265 of the 1459 team-seasons, so using only the NHL keeps most of the data.
- The NHL has 92 seasons between 1917 and 2009. 2004 is missing because of the lockout, so a "previous season" feature can't just use `year - 1`.
- Master has 7236 player ids and 392 coach ids, and 268 of them are both.
- The NHL has 55 team codes (`tmID`) but only 36 franchises (`franchID`), so a team changes code over time. We look at this in section 3.
- Missing values are high in Master (39%), Scoring (34%), TeamSplits (33%) and Goalies (31%). Section 5 looks at how this depends on the era.

## 3. Keys and relationships

FINDING the key of each table and check the links between them: `tmID` + `year` links teams, coaches and scoring; `playerID` and `coachID` link to `Master`.

In [48]:
def is_key(df, cols):
    # True if cols uniquely identify every row and contain no missing values.
    return (not df[cols].isna().any().any()) and (not df.duplicated(subset=cols).any())

candidate_keys = {
    "Teams": ["year", "tmID"],
    "Coaches": ["coachID", "year", "tmID", "stint"],
    "Scoring": ["playerID", "year", "stint"],
    "Goalies": ["playerID", "year", "stint"],
    "TeamsHalf": ["year", "tmID", "half"],
    "TeamsPost": ["year", "tmID"],
}
pd.Series({t: is_key(tables[t], k) for t, k in candidate_keys.items()}, name="is_key")

Teams        True
Coaches      True
Scoring      True
Goalies      True
TeamsHalf    True
TeamsPost    True
Name: is_key, dtype: bool

## 4. Coverage and league evolution

This section justifies the scope decision (which leagues, which era).


## 5. Data quality
